# Finite Hermite mixed-sampling experiment

This notebook generates paper Figure 1: the relative smallest singular value of the mixed 3 x 3 sampling matrix in the Hermite space, with one time sample fixed at zero and two frequency samples varied over a square grid.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = Path("results/finite_hermite")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Experiment configuration
d = 3
t0 = 0.0
scan_limit = 4.0
num_points = 16001
sv_rel_tol = 1.85e-5


def build_hermite_matrix(points, dimension):
    """Evaluate normalized Hermite functions phi_0, ..., phi_(dimension-1)."""
    points = np.atleast_1d(np.asarray(points, dtype=float))
    matrix = np.empty((points.size, dimension), dtype=float)

    phi_previous = np.pi ** (-0.25) * np.exp(-0.5 * points**2)
    matrix[:, 0] = phi_previous
    if dimension == 1:
        return matrix

    phi_current = np.sqrt(2.0) * points * phi_previous
    matrix[:, 1] = phi_current
    for n in range(1, dimension - 1):
        phi_next = (
            np.sqrt(2.0 / (n + 1)) * points * phi_current
            - np.sqrt(n / (n + 1)) * phi_previous
        )
        matrix[:, n + 1] = phi_next
        phi_previous, phi_current = phi_current, phi_next

    return matrix


omega = np.linspace(-scan_limit, scan_limit, num_points)
phase_factors = (-1j) ** np.arange(d)
frequency_rows = build_hermite_matrix(omega, d) * phase_factors
time_row = build_hermite_matrix([t0], d)[0].astype(complex)

# The result is symmetric under swapping omega_0 and omega_1, so compute
# only the upper triangle and mirror it into the lower triangle.
log10_ratio = np.empty((num_points, num_points), dtype=float)

for i in range(num_points):
    count = num_points - i
    matrices = np.empty((count, d, d), dtype=complex)
    matrices[:, 0, :] = time_row
    matrices[:, 1, :] = frequency_rows[i]
    matrices[:, 2, :] = frequency_rows[i:]

    singular_values = np.linalg.svd(matrices, compute_uv=False)
    ratios = singular_values[:, -1] / np.maximum(
        singular_values[:, 0], np.finfo(float).tiny
    )
    values = np.log10(np.maximum(ratios, 1e-300))

    log10_ratio[i, i:] = values
    log10_ratio[i:, i] = values

print(f"Computed {num_points} x {num_points} singular-value grid.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.2))
image = ax.imshow(
    log10_ratio.T,
    origin="lower",
    extent=[omega.min(), omega.max(), omega.min(), omega.max()],
    aspect="auto",
    cmap="gray",
)

ax.set_xlabel(r"$\omega_0$", fontsize=26)
ax.set_ylabel(r"$\omega_1$", fontsize=26)
ax.contour(
    omega,
    omega,
    log10_ratio.T,
    levels=[np.log10(sv_rel_tol)],
    linewidths=3.0,
)
ax.scatter([1.0, -1.0], [-1.0, 1.0], marker="x")

colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label(
    r"$\log_{10}(\sigma_{\min}/\sigma_{\max})$", fontsize=20
)

fig.tight_layout()
output_path = RESULTS_DIR / "mixed_sampling.png"
fig.savefig(output_path, dpi=150)
plt.show()

index_pos = np.argmin(np.abs(omega - 1.0))
index_neg = np.argmin(np.abs(omega + 1.0))
print(
    "(omega_0, omega_1)=(1, -1): ",
    f"log10(sigma_min/sigma_max)={log10_ratio[index_pos, index_neg]:.6f}",
)
print(f"Saved figure to {output_path}")